# Hypothesis Testing — Tutorial

**Primary outcome:** Formulate testable hypotheses, choose the right statistical test, interpret p-values and effect sizes, and design an A/B test using real-world data.

**Time:** 60–90 minutes

---

## What you will learn

- Write null and alternative hypotheses for real data questions
- Select the correct test based on data type and group count
- Run one-sample t-tests, two-sample t-tests, Mann-Whitney U, ANOVA, and chi-square tests
- Calculate Cohen's d and interpret effect sizes
- Simulate and evaluate an A/B test end-to-end
- Compute required sample sizes with power analysis

---

## Datasets

| Dataset | Source | Contents |
|---------|--------|----------|
| `tips` | `seaborn.load_dataset('tips')` | Restaurant bills, tips, gender, day, time, party size |
| `titanic` | `seaborn.load_dataset('titanic')` | Passenger survival, class, age, fare, sex |

## Section 1 — The Scientific Method in Data

### The hypothesis-test-conclude cycle

Every good data experiment follows the same five steps:

1. **Observe** — Notice a pattern or question in the data
2. **Hypothesise** — State what you expect to be true (and what you expect *not* to be true)
3. **Design** — Choose the right test, set your significance level (α), decide on sample size
4. **Test** — Collect/use data, run the statistical test, compute the p-value
5. **Conclude** — Compare p-value to α, decide whether to reject H₀, state the real-world meaning

```
Observe → Hypothesise → Design → Test → Conclude → (back to Observe)
```

---

### Null vs Alternative hypothesis

| Symbol | Name | Plain English |
|--------|------|---------------|
| **H₀** | Null hypothesis | The boring default — "nothing is going on", no effect, no difference |
| **H₁** (or Hₐ) | Alternative hypothesis | The claim you actually want to test — "something *is* going on" |

A statistical test does **not** prove H₁ is true. It tells you whether the data are *inconsistent enough* with H₀ to justify rejecting it.

---

### Type I and Type II errors

| | H₀ is actually TRUE | H₀ is actually FALSE |
|---|---|---|
| **Reject H₀** | ❌ **Type I error** (false positive) — probability = α | ✅ Correct (power = 1 − β) |
| **Fail to reject H₀** | ✅ Correct | ❌ **Type II error** (false negative) — probability = β |

**Consequences in practice:**

- **Type I error (α):** You launch a feature that actually does nothing. Wasted engineering effort, misleading stakeholders.
- **Type II error (β):** You miss a real improvement and don't ship a genuinely better product.

The conventional threshold is **α = 0.05** (5% chance of a false positive). Medical trials often use α = 0.01 because a Type I error could harm patients.

## Section 2 — Formulating Hypotheses

We will use three questions about the `tips` dataset to practise writing H₀ and H₁.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.stats.multicomp as mc
import statsmodels.stats.power as smp
from statsmodels.stats.proportion import proportions_ztest

np.random.seed(42)
sns.set_theme(style='whitegrid', palette='muted')

# Load datasets
tips    = sns.load_dataset('tips')
titanic = sns.load_dataset('titanic')

print('tips shape:   ', tips.shape)
print('titanic shape:', titanic.shape)
tips.head()

In [ ]:
# --- Hypothesis Example (a) ---
# Question: Are dinner bills higher than lunch bills?

print('=== Example (a): Dinner bills vs Lunch bills ===')
print('H0: The mean total_bill at dinner is EQUAL TO the mean total_bill at lunch.')
print('    μ_dinner = μ_lunch')
print('H1: The mean total_bill at dinner is GREATER THAN the mean total_bill at lunch.')
print('    μ_dinner > μ_lunch  (one-tailed, upper)')

dinner = tips.loc[tips['time'] == 'Dinner', 'total_bill']
lunch  = tips.loc[tips['time'] == 'Lunch',  'total_bill']
print(f'\nSample means → Dinner: ${dinner.mean():.2f}  |  Lunch: ${lunch.mean():.2f}')

# --- Hypothesis Example (b) ---
print('\n=== Example (b): Tipping rate by gender ===')
print('H0: The mean tipping rate (tip / total_bill) is the SAME for male and female customers.')
print('    μ_male_rate = μ_female_rate')
print('H1: The mean tipping rate DIFFERS between male and female customers.')
print('    μ_male_rate ≠ μ_female_rate  (two-tailed)')

tips['tip_rate'] = tips['tip'] / tips['total_bill']
male   = tips.loc[tips['sex'] == 'Male',   'tip_rate']
female = tips.loc[tips['sex'] == 'Female', 'tip_rate']
print(f'\nSample means → Male: {male.mean():.4f}  |  Female: {female.mean():.4f}')

# --- Hypothesis Example (c) ---
print('\n=== Example (c): Party size and bill amount ===')
print('H0: Party size has NO EFFECT on total_bill.')
print('    All group means are equal  (ANOVA null hypothesis)')
print('H1: At least one party size group has a DIFFERENT mean total_bill.')
print('    At least one μ_i ≠ μ_j')
print(f'\nParty sizes observed: {sorted(tips["size"].unique())}')

## Section 3 — Choosing the Right Test

Use this decision tree before running any test. Ask yourself three questions:

1. What **type of data** do I have? (continuous / categorical)
2. How many **groups** am I comparing?
3. Is the data **normally distributed**? (affects parametric vs non-parametric choice)

| Data type | Groups | Normal? | Recommended test |
|-----------|--------|---------|------------------|
| Continuous | 1 (vs known mean) | Yes | **One-sample t-test** |
| Continuous | 2 independent | Yes | **Two-sample (independent) t-test** |
| Continuous | 2 paired (before/after) | Yes | **Paired t-test** |
| Continuous | 2 independent | No / skewed | **Mann-Whitney U** |
| Continuous | 3 or more | Yes | **One-way ANOVA** (+ post-hoc Tukey) |
| Continuous | 3 or more | No / skewed | **Kruskal-Wallis** |
| Categorical | 2 variables (independence) | N/A | **Chi-square test** |
| Proportion | 2 groups | N/A | **Z-test for proportions** |

> **Rule of thumb for normality:** n > 30 per group usually satisfies the Central Limit Theorem regardless of the underlying distribution. For small samples, run a Shapiro-Wilk test first.

## Section 4 — One-Sample t-Test

**Question:** Does the mean tip in our dataset differ from the restaurant industry standard of $2.50?

- **H₀:** μ_tip = $2.50
- **H₁:** μ_tip ≠ $2.50 (two-tailed)
- **α = 0.05**

In [ ]:
industry_standard = 2.50
sample_mean = tips['tip'].mean()

t_stat, p_value = stats.ttest_1samp(tips['tip'], popmean=industry_standard)

print('=== One-Sample t-Test ===')
print(f'Industry standard (H0 mean): ${industry_standard:.2f}')
print(f'Sample mean:                 ${sample_mean:.4f}')
print(f'Sample size:                 n = {len(tips)}')
print(f't-statistic:                 {t_stat:.4f}')
print(f'p-value:                     {p_value:.4f}')
print()

alpha = 0.05
if p_value < alpha:
    print(f'Result: p ({p_value:.4f}) < α ({alpha}) → REJECT H0')
    print(f'Conclusion: The mean tip (${sample_mean:.2f}) is statistically different from the '
          f'industry standard of ${industry_standard:.2f}.')
else:
    print(f'Result: p ({p_value:.4f}) >= α ({alpha}) → FAIL TO REJECT H0')
    print('Conclusion: No significant difference from the industry standard.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(tips['tip'], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(industry_standard, color='crimson',   linestyle='--', linewidth=2,
           label=f'H₀ mean (industry standard) = ${industry_standard}')
ax.axvline(sample_mean,       color='darkorange', linestyle='-',  linewidth=2,
           label=f'Sample mean = ${sample_mean:.2f}')

ax.set_title('Distribution of Tips — One-Sample t-Test', fontsize=14)
ax.set_xlabel('Tip ($)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print('The orange line (sample mean) sits visibly to the right of the red dashed line (H0).')
print('The t-test confirms this gap is statistically significant.')

## Section 5 — Two-Sample t-Test

**Question:** Do male and female customers tip at different rates?

Before running a t-test we should verify the normality assumption with a **Shapiro-Wilk test**.

- **H₀ (Shapiro-Wilk):** The data comes from a normal distribution
- **H₁ (Shapiro-Wilk):** The data does NOT come from a normal distribution

In [ ]:
# Normality check first
sw_male,   p_male   = stats.shapiro(male)
sw_female, p_female = stats.shapiro(female)

print('=== Shapiro-Wilk Normality Test ===')
print(f'Male tip rate   → W = {sw_male:.4f}, p = {p_male:.4f}')
print(f'Female tip rate → W = {sw_female:.4f}, p = {p_female:.4f}')
print()

for label, p in [('Male', p_male), ('Female', p_female)]:
    if p < 0.05:
        print(f'{label}: p < 0.05 → distribution is NOT normal')
    else:
        print(f'{label}: p >= 0.05 → distribution appears normal')

In [ ]:
# Two-sample independent t-test (Welch's — does not assume equal variances)
t_stat, p_value = stats.ttest_ind(male, female, equal_var=False)

print('=== Two-Sample (Welch) t-Test: tip rate Male vs Female ===')
print(f'Male mean:   {male.mean():.4f}  (n={len(male)})')
print(f'Female mean: {female.mean():.4f}  (n={len(female)})')
print(f't-statistic: {t_stat:.4f}')
print(f'p-value:     {p_value:.4f}')
print()

alpha = 0.05
if p_value < alpha:
    print(f'Result: p ({p_value:.4f}) < α ({alpha}) → REJECT H0')
    print('Conclusion: There IS a statistically significant difference in tipping rate between genders.')
else:
    print(f'Result: p ({p_value:.4f}) >= α ({alpha}) → FAIL TO REJECT H0')
    print('Conclusion: No statistically significant difference in tipping rate between genders.')

# --- Cohen's d ---
def cohens_d(group1, group2):
    """Pooled Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    pooled_std = np.sqrt(((n1 - 1) * group1.std(ddof=1)**2 +
                          (n2 - 1) * group2.std(ddof=1)**2) / (n1 + n2 - 2))
    return (group1.mean() - group2.mean()) / pooled_std

d = cohens_d(male, female)
print(f"\nCohen's d: {d:.4f}")
if abs(d) < 0.2:
    print('Effect size interpretation: negligible')
elif abs(d) < 0.5:
    print('Effect size interpretation: small')
elif abs(d) < 0.8:
    print('Effect size interpretation: medium')
else:
    print('Effect size interpretation: large')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tips.boxplot(column='tip_rate', by='sex', ax=ax, grid=False)
ax.set_title('Tip Rate by Gender')
ax.set_xlabel('Gender')
ax.set_ylabel('Tip Rate (tip / total_bill)')
plt.suptitle('')
plt.tight_layout()
plt.show()

### Statistical significance vs practical significance

A result can be **statistically significant** (p < 0.05) without being **practically significant** (Cohen's d is tiny).

| Scenario | p-value | Cohen's d | Verdict |
|----------|---------|-----------|--------|
| Large sample, trivial difference | < 0.05 | 0.05 | Statistically significant, but practically meaningless |
| Small sample, real difference | 0.12 | 0.70 | Not statistically significant *yet*, but could matter — collect more data |
| The sweet spot | < 0.05 | 0.65 | Statistically AND practically significant |

> **Takeaway:** Always report both the p-value **and** an effect size. A p-value alone does not tell you whether the difference matters in the real world.

## Section 6 — Mann-Whitney U Test (Non-Parametric)

When data is heavily skewed or the normality assumption cannot be met, use **Mann-Whitney U** instead of a t-test.

It tests whether values from one group tend to be **ranked higher** than values from another group — no normality required.

We will use `total_bill` (which is right-skewed) split by time of day.

In [ ]:
bill_dinner = tips.loc[tips['time'] == 'Dinner', 'total_bill']
bill_lunch  = tips.loc[tips['time'] == 'Lunch',  'total_bill']

# Check skewness
print('Skewness of total_bill (Dinner):', bill_dinner.skew().round(3))
print('Skewness of total_bill (Lunch): ', bill_lunch.skew().round(3))
print('(Values > 1 indicate notable right skew)')

# Mann-Whitney U
u_stat, p_mw = stats.mannwhitneyu(bill_dinner, bill_lunch, alternative='two-sided')
print(f'\n=== Mann-Whitney U Test ===')
print(f'U-statistic: {u_stat:.1f}')
print(f'p-value:     {p_mw:.4f}')

# Compare with t-test for contrast
t_stat2, p_t2 = stats.ttest_ind(bill_dinner, bill_lunch, equal_var=False)
print(f'\nFor comparison — Welch t-test p-value: {p_t2:.4f}')
print()

alpha = 0.05
if p_mw < alpha:
    print(f'Mann-Whitney: p ({p_mw:.4f}) < α ({alpha}) → REJECT H0')
    print('Conclusion: Dinner bills tend to be significantly higher than lunch bills.')
else:
    print(f'Mann-Whitney: p ({p_mw:.4f}) >= α ({alpha}) → FAIL TO REJECT H0')
    print('Conclusion: No significant difference in bill amounts by meal time.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Distribution plot showing skew
axes[0].hist(bill_dinner, bins=15, alpha=0.6, color='teal',   label='Dinner')
axes[0].hist(bill_lunch,  bins=15, alpha=0.6, color='salmon', label='Lunch')
axes[0].set_title('Distribution of Total Bill by Time')
axes[0].set_xlabel('Total Bill ($)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Box plot
tips.boxplot(column='total_bill', by='time', ax=axes[1], grid=False)
axes[1].set_title('Box Plot: Total Bill by Time')
axes[1].set_xlabel('Time of Day')
axes[1].set_ylabel('Total Bill ($)')
plt.suptitle('')

plt.tight_layout()
plt.show()
print('Note the right skew in the histogram — this is why Mann-Whitney U is preferred here.')

## Section 7 — ANOVA: Comparing More Than Two Groups

**Question:** Does mean total_bill differ across the four days (Thu, Fri, Sat, Sun)?

- **H₀:** μ_Thu = μ_Fri = μ_Sat = μ_Sun
- **H₁:** At least one day has a different mean total_bill
- **Test:** One-way ANOVA (F-test)

If ANOVA is significant, we need a **post-hoc test** to find *which* pairs differ — we use Tukey HSD.

In [ ]:
groups = [tips.loc[tips['day'] == day, 'total_bill'] for day in ['Thur', 'Fri', 'Sat', 'Sun']]

f_stat, p_anova = stats.f_oneway(*groups)

print('=== One-Way ANOVA: Total Bill by Day ===')
print(f'F-statistic: {f_stat:.4f}')
print(f'p-value:     {p_anova:.4f}')
print()

alpha = 0.05
if p_anova < alpha:
    print(f'Result: p ({p_anova:.4f}) < α ({alpha}) → REJECT H0')
    print('Conclusion: At least one day has a significantly different mean bill.')
    print('Running post-hoc Tukey HSD to find which pairs differ...')
else:
    print(f'Result: p ({p_anova:.4f}) >= α ({alpha}) → FAIL TO REJECT H0')
    print('Conclusion: No significant difference in mean bill across days.')

In [ ]:
# Post-hoc Tukey HSD
tukey_result = mc.pairwise_tukeyhsd(
    endog=tips['total_bill'],
    groups=tips['day'],
    alpha=0.05
)
print('=== Tukey HSD Post-Hoc Test ===')
print(tukey_result)
print()
print('Pairs where reject=True have statistically different mean bills.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
tips.boxplot(column='total_bill', by='day', ax=ax,
             order=['Thur', 'Fri', 'Sat', 'Sun'], grid=False)
ax.set_title('Total Bill by Day of Week (ANOVA)')
ax.set_xlabel('Day')
ax.set_ylabel('Total Bill ($)')
plt.suptitle('')
plt.tight_layout()
plt.show()

## Section 8 — Chi-Square Test of Independence

**Question:** Is passenger survival independent of ticket class on the Titanic?

- **H₀:** Survival and passenger class are independent (no relationship)
- **H₁:** Survival and passenger class are NOT independent
- **Test:** Chi-square (χ²) test of independence

Chi-square works with **categorical counts** — it compares what we *observed* to what we'd *expect* if the two variables were independent.

In [ ]:
titanic_clean = titanic.dropna(subset=['survived', 'pclass'])

# Contingency table
contingency = pd.crosstab(titanic_clean['pclass'], titanic_clean['survived'],
                           rownames=['Passenger Class'],
                           colnames=['Survived (0=No, 1=Yes)'])
print('=== Observed Counts ===')
print(contingency)

chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)

print('\n=== Expected Counts (if independent) ===')
expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)
print(expected_df.round(1))

print(f'\nChi-square statistic: {chi2:.4f}')
print(f'Degrees of freedom:   {dof}')
print(f'p-value:              {p_chi2:.6f}')
print()

alpha = 0.05
if p_chi2 < alpha:
    print(f'Result: p ({p_chi2:.6f}) < α ({alpha}) → REJECT H0')
    print('Conclusion: Survival is NOT independent of passenger class.')
    print('First-class passengers had significantly higher survival rates.')
else:
    print(f'Result: p ({p_chi2:.6f}) >= α ({alpha}) → FAIL TO REJECT H0')
    print('Conclusion: No significant association between class and survival.')

In [ ]:
# Heatmap of Pearson residuals (observed - expected) / sqrt(expected)
residuals = (contingency.values - expected) / np.sqrt(expected)
residuals_df = pd.DataFrame(residuals, index=contingency.index, columns=contingency.columns)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(contingency, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Observed Counts')

sns.heatmap(residuals_df, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1])
axes[1].set_title('Pearson Residuals\n(positive = more than expected, negative = less)')

plt.tight_layout()
plt.show()
print('Large positive residuals → survived more than chance predicts.')
print('Large negative residuals → survived less than chance predicts.')

## Section 9 — A/B Testing End-to-End

### Scenario

You are a data scientist at an e-commerce company. The current checkout page (**control**) has a 5% conversion rate. A redesigned page (**treatment**) is expected to push this to 6.5%. You ran both versions for two weeks with 1000 users each. Did the redesign actually work?

- **H₀:** p_control = p_treatment (conversion rates are equal)
- **H₁:** p_treatment > p_control (one-tailed — we care about improvements)
- **α = 0.05**

In [ ]:
np.random.seed(42)

n_control   = 1000
n_treatment = 1000
p_control   = 0.05
p_treatment = 0.065

# Simulate binary outcomes (1 = converted, 0 = did not convert)
control_outcomes   = np.random.binomial(1, p_control,   n_control)
treatment_outcomes = np.random.binomial(1, p_treatment, n_treatment)

conv_control   = control_outcomes.sum()
conv_treatment = treatment_outcomes.sum()

obs_rate_control   = conv_control   / n_control
obs_rate_treatment = conv_treatment / n_treatment

print('=== A/B Test Data ===')
print(f'Control:   {conv_control} conversions / {n_control} users = {obs_rate_control:.2%}')
print(f'Treatment: {conv_treatment} conversions / {n_treatment} users = {obs_rate_treatment:.2%}')
print(f'Lift:      {(obs_rate_treatment - obs_rate_control) / obs_rate_control:.1%}')

# Two-proportions z-test (one-tailed)
count  = np.array([conv_treatment, conv_control])
nobs   = np.array([n_treatment, n_control])
z_stat, p_z = proportions_ztest(count, nobs, alternative='larger')

print(f'\n=== Two-Proportions Z-Test (one-tailed) ===')
print(f'z-statistic: {z_stat:.4f}')
print(f'p-value:     {p_z:.4f}')
print()

alpha = 0.05
if p_z < alpha:
    print(f'Result: p ({p_z:.4f}) < α ({alpha}) → REJECT H0')
    print('Conclusion: The treatment page has a significantly higher conversion rate.')
    print('Recommendation: Ship the redesign.')
else:
    print(f'Result: p ({p_z:.4f}) >= α ({alpha}) → FAIL TO REJECT H0')
    print('Conclusion: The improvement is NOT statistically significant.')
    print('Recommendation: Run the test longer or collect more data.')

In [ ]:
# Confidence intervals for the two proportions
from statsmodels.stats.proportion import proportion_confint

ci_control   = proportion_confint(conv_control,   n_control,   alpha=0.05, method='wilson')
ci_treatment = proportion_confint(conv_treatment, n_treatment, alpha=0.05, method='wilson')

labels = ['Control', 'Treatment']
means  = [obs_rate_control, obs_rate_treatment]
lo     = [obs_rate_control   - ci_control[0],   obs_rate_treatment   - ci_treatment[0]]
hi     = [ci_control[1]   - obs_rate_control,   ci_treatment[1]   - obs_rate_treatment]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, means, color=['steelblue', 'darkorange'],
       yerr=[lo, hi], capsize=10, width=0.4, edgecolor='white')
ax.set_ylabel('Conversion Rate')
ax.set_title('A/B Test: Conversion Rates with 95% CI')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1%}'))
for i, (m, label) in enumerate(zip(means, labels)):
    ax.text(i, m + 0.003, f'{m:.2%}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(f'Control 95% CI:   [{ci_control[0]:.3%}, {ci_control[1]:.3%}]')
print(f'Treatment 95% CI: [{ci_treatment[0]:.3%}, {ci_treatment[1]:.3%}]')
print('Overlapping CIs do NOT necessarily mean the test is non-significant — use the z-test for decisions.')

## Section 10 — Effect Size and Practical Significance

Cohen's d benchmarks (Cohen, 1988):

| d | Interpretation |
|---|----------------|
| 0.2 | Small |
| 0.5 | Medium |
| 0.8 | Large |

Below we compare two scenarios where p < 0.05, but with very different practical implications.

In [ ]:
np.random.seed(42)

# Scenario A: statistically significant, but tiny effect (d ≈ 0.1)
n_large = 5000
group_a1 = np.random.normal(loc=100.0, scale=10, size=n_large)
group_a2 = np.random.normal(loc=101.0, scale=10, size=n_large)  # only 1-unit difference

t_a, p_a = stats.ttest_ind(group_a1, group_a2)
d_a = cohens_d(pd.Series(group_a1), pd.Series(group_a2))

# Scenario B: statistically significant AND large effect (d ≈ 0.8)
n_small = 200
group_b1 = np.random.normal(loc=100, scale=10, size=n_small)
group_b2 = np.random.normal(loc=108, scale=10, size=n_small)  # 8-unit difference

t_b, p_b = stats.ttest_ind(group_b1, group_b2)
d_b = cohens_d(pd.Series(group_b1), pd.Series(group_b2))

print('=== Scenario A: Tiny effect, large sample ===')
print(f'n = {n_large} per group | mean difference = {group_a2.mean() - group_a1.mean():.2f}')
print(f'p-value: {p_a:.4f} | Cohen\'s d: {d_a:.3f} (small effect)')
print()

print('=== Scenario B: Large effect, smaller sample ===')
print(f'n = {n_small} per group | mean difference = {group_b2.mean() - group_b1.mean():.2f}')
print(f'p-value: {p_b:.4f} | Cohen\'s d: {d_b:.3f} (large effect)')
print()
print('Both are statistically significant, but Scenario B represents a meaningful real-world change.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, g1, g2, title, d in [
    (axes[0], group_a1, group_a2, f'Scenario A — d = {d_a:.2f} (small)', d_a),
    (axes[1], group_b1, group_b2, f'Scenario B — d = {d_b:.2f} (large)', d_b),
]:
    x_min = min(g1.min(), g2.min()) - 5
    x_max = max(g1.max(), g2.max()) + 5
    xs = np.linspace(x_min, x_max, 300)
    ax.plot(xs, stats.norm.pdf(xs, g1.mean(), g1.std()), color='steelblue',   label='Group 1')
    ax.plot(xs, stats.norm.pdf(xs, g2.mean(), g2.std()), color='darkorange', label='Group 2')
    ax.axvline(g1.mean(), color='steelblue',   linestyle='--', alpha=0.6)
    ax.axvline(g2.mean(), color='darkorange',  linestyle='--', alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle("Statistical vs Practical Significance", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Section 11 — Power Analysis and Sample Size

**Statistical power** = the probability of correctly rejecting H₀ when it is false (1 − β).

Standard targets: **power ≥ 0.80** with α = 0.05.

Power depends on three things:
- **Effect size** — bigger effect → easier to detect → need fewer samples
- **Sample size** — more data → more power
- **α** — higher α (less strict) → more power, but more false positives

### Practical question

You are running the A/B test from Section 9 again. How many users do you need per group to detect a lift from 5% to 6.5% with 80% power?

In [ ]:
from statsmodels.stats.power import TTestIndPower

power_analysis = TTestIndPower()

# How many samples do we need for different effect sizes at 80% power?
effect_sizes = [0.2, 0.5, 0.8]  # small, medium, large

print('=== Required Sample Size per Group (α=0.05, power=0.80) ===')
print(f'{"Effect Size":<15} {"Cohen\'s d":<12} {"n per group":<12}')
print('-' * 40)
for label, d_val in zip(['Small', 'Medium', 'Large'], effect_sizes):
    n_required = power_analysis.solve_power(
        effect_size=d_val, alpha=0.05, power=0.80, alternative='two-sided'
    )
    print(f'{label:<15} {d_val:<12.1f} {int(np.ceil(n_required)):<12}')

# For our specific A/B test — conversion 5% vs 6.5%
# Effect size for proportions: Cohen's h
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

h = proportion_effectsize(0.065, 0.05)
n_ab = NormalIndPower().solve_power(effect_size=h, alpha=0.05, power=0.80, alternative='larger')
print(f'\nFor our A/B test (5% → 6.5% conversion):')
print(f'Cohen\'s h: {h:.4f}')
print(f'Required n per group: {int(np.ceil(n_ab))} users')
print(f'\nWith only {n_control} users per group we had this power:')
power_actual = NormalIndPower().solve_power(
    effect_size=h, alpha=0.05, nobs1=n_control, alternative='larger'
)
print(f'Actual power: {power_actual:.2%}')

In [ ]:
# Power curves: power vs sample size for different effect sizes
sample_sizes = np.arange(10, 500, 5)

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['tomato', 'steelblue', 'seagreen']
for d_val, label, color in zip([0.2, 0.5, 0.8], ['Small (d=0.2)', 'Medium (d=0.5)', 'Large (d=0.8)'], colors):
    powers = [
        power_analysis.solve_power(
            effect_size=d_val, nobs1=n, alpha=0.05, alternative='two-sided'
        )
        for n in sample_sizes
    ]
    ax.plot(sample_sizes, powers, label=label, color=color, linewidth=2)

ax.axhline(0.80, color='black', linestyle='--', linewidth=1.5, label='80% power threshold')
ax.set_title('Power Curves for Different Effect Sizes', fontsize=14)
ax.set_xlabel('Sample Size per Group')
ax.set_ylabel('Statistical Power')
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('To detect a small effect you need many more participants than for a large effect.')
print('Always run a power analysis BEFORE collecting data — not after.')

## Section 12 — Practice Exercises

Work through the three exercises below. Each has a blank cell for your solution followed by a hidden solution cell.

---

### Exercise 1: Paired t-Test (Before/After)

A company runs a training programme. Below is synthetic data for 30 employees' productivity scores before and after training.

**Your task:**
1. State H₀ and H₁
2. Run a **paired t-test** (`scipy.stats.ttest_rel`)
3. Report the t-statistic, p-value, and conclusion
4. Calculate Cohen's d for paired data: `d = mean(diff) / std(diff)`

In [ ]:
np.random.seed(42)
before = np.random.normal(loc=70, scale=8, size=30)
after  = before + np.random.normal(loc=5, scale=4, size=30)  # training adds ~5 points

# Your code here
# 1. Print H0 and H1
# 2. Run stats.ttest_rel(before, after)
# 3. Print t-stat, p-value, conclusion
# 4. Compute Cohen's d for paired data

In [ ]:
# === SOLUTION — Exercise 1 ===

print('H0: Training has NO effect on productivity. μ_before = μ_after')
print('H1: Training IMPROVES productivity. μ_after > μ_before (one-tailed)')
print()

t_paired, p_paired = stats.ttest_rel(before, after)
diff = after - before
d_paired = diff.mean() / diff.std(ddof=1)

print(f'Mean productivity before: {before.mean():.2f}')
print(f'Mean productivity after:  {after.mean():.2f}')
print(f'Mean improvement:         {diff.mean():.2f} points')
print(f't-statistic: {t_paired:.4f}')
# ttest_rel gives a two-tailed p; halve for one-tailed (same direction)
print(f'p-value (two-tailed): {p_paired:.4f}')
print(f'p-value (one-tailed): {p_paired / 2:.4f}')
print(f"Cohen's d (paired):   {d_paired:.4f}")
print()

if p_paired / 2 < 0.05:
    print('Result: REJECT H0')
    print('Conclusion: Training significantly improved productivity.')
    if abs(d_paired) >= 0.8:
        print(f"Effect size d = {d_paired:.2f} → large practical effect.")
else:
    print('Result: FAIL TO REJECT H0')

---

### Exercise 2: ANOVA on Titanic Fare by Passenger Class

**Your task:**
1. Using the `titanic` dataset, test whether mean `fare` differs across the three passenger classes (pclass 1, 2, 3)
2. State H₀ and H₁
3. Run `scipy.stats.f_oneway` and interpret the result
4. If significant, run Tukey HSD post-hoc and identify which class pairs differ

In [ ]:
# Your code here
# Hint: filter titanic by pclass 1, 2, 3 and use the 'fare' column
# titanic.dropna(subset=['fare', 'pclass']) first

In [ ]:
# === SOLUTION — Exercise 2 ===

print('H0: Mean fare is the same across all three passenger classes.')
print('    μ_class1 = μ_class2 = μ_class3')
print('H1: At least one class has a different mean fare.')
print()

titanic_fare = titanic.dropna(subset=['fare', 'pclass'])

cls1 = titanic_fare.loc[titanic_fare['pclass'] == 1, 'fare']
cls2 = titanic_fare.loc[titanic_fare['pclass'] == 2, 'fare']
cls3 = titanic_fare.loc[titanic_fare['pclass'] == 3, 'fare']

print(f'Class 1 — mean: £{cls1.mean():.2f}  n={len(cls1)}')
print(f'Class 2 — mean: £{cls2.mean():.2f}  n={len(cls2)}')
print(f'Class 3 — mean: £{cls3.mean():.2f}  n={len(cls3)}')

f_stat, p_anova2 = stats.f_oneway(cls1, cls2, cls3)
print(f'\nF-statistic: {f_stat:.4f}  |  p-value: {p_anova2:.6f}')

if p_anova2 < 0.05:
    print('REJECT H0 — fare differs significantly across classes.')
    print('\n=== Tukey HSD Post-Hoc ===')
    tukey2 = mc.pairwise_tukeyhsd(
        endog=titanic_fare['fare'],
        groups=titanic_fare['pclass'],
        alpha=0.05
    )
    print(tukey2)
    print('\nAll three class pairs differ significantly in fare.')
else:
    print('FAIL TO REJECT H0.')

---

### Exercise 3: Design an A/B Test for a Hypothetical App

**Scenario:** You are a product analyst at a mobile app. The current onboarding flow has a 12% completion rate. You believe a shorter onboarding (fewer screens) will increase this to 16%.

**Your task:**
1. State H₀ and H₁
2. Choose the significance level and desired power
3. Compute the required sample size per group using `NormalIndPower` and `proportion_effectsize`
4. Simulate the experiment and run a proportions z-test
5. Write a 2-sentence business recommendation based on your result

In [ ]:
# Your code here
# Hint: proportion_effectsize(0.16, 0.12) gives Cohen's h
# NormalIndPower().solve_power(effect_size=h, alpha=..., power=...) gives n

In [ ]:
# === SOLUTION — Exercise 3 ===

print('=== Step 1: Hypotheses ===')
print('H0: Short onboarding completion rate = long onboarding rate  (p_short = p_long = 0.12)')
print('H1: Short onboarding has a HIGHER completion rate  (p_short > p_long)')
print()

print('=== Step 2: Parameters ===')
alpha_ex3 = 0.05
power_ex3 = 0.80
p_ctrl_ex3  = 0.12
p_treat_ex3 = 0.16
print(f'α = {alpha_ex3}  |  target power = {power_ex3}')
print()

print('=== Step 3: Sample Size ===')
h_ex3 = proportion_effectsize(p_treat_ex3, p_ctrl_ex3)
n_ex3 = NormalIndPower().solve_power(
    effect_size=h_ex3, alpha=alpha_ex3, power=power_ex3, alternative='larger'
)
n_ex3 = int(np.ceil(n_ex3))
print(f"Cohen's h: {h_ex3:.4f}")
print(f'Required n per group: {n_ex3} users')
print()

print('=== Step 4: Simulate and Test ===')
np.random.seed(42)
ctrl_conv_ex3  = np.random.binomial(1, p_ctrl_ex3,  n_ex3).sum()
treat_conv_ex3 = np.random.binomial(1, p_treat_ex3, n_ex3).sum()

z_ex3, p_ex3 = proportions_ztest(
    [treat_conv_ex3, ctrl_conv_ex3], [n_ex3, n_ex3], alternative='larger'
)
print(f'Control:   {ctrl_conv_ex3}/{n_ex3} = {ctrl_conv_ex3/n_ex3:.2%}')
print(f'Treatment: {treat_conv_ex3}/{n_ex3} = {treat_conv_ex3/n_ex3:.2%}')
print(f'z-stat: {z_ex3:.4f}  |  p-value: {p_ex3:.4f}')
print()

print('=== Step 5: Business Recommendation ===')
if p_ex3 < alpha_ex3:
    print('The shorter onboarding flow produced a statistically significant improvement in '
          'completion rate, from 12% to approximately 16%.')
    print('We recommend shipping the shorter onboarding to all users, which could '
          f'meaningfully increase the number of activated users per cohort.')
else:
    print('The data did not provide sufficient evidence that the shorter onboarding improves completion rate.')
    print('We recommend extending the test duration or exploring a more impactful redesign before rolling out.')

---

## Summary

| Section | Key concept | Tool used |
|---------|-------------|----------|
| 1 | Null/alternative hypothesis, Type I/II errors | Conceptual |
| 2 | Writing H₀ and H₁ for real questions | `tips` dataset |
| 3 | Choosing the right test | Decision tree |
| 4 | One-sample t-test | `scipy.stats.ttest_1samp` |
| 5 | Two-sample t-test, Cohen's d | `scipy.stats.ttest_ind`, `shapiro` |
| 6 | Non-parametric alternative | `scipy.stats.mannwhitneyu` |
| 7 | Comparing 3+ groups | `scipy.stats.f_oneway`, Tukey HSD |
| 8 | Categorical independence | `scipy.stats.chi2_contingency` |
| 9 | A/B test end-to-end | `proportions_ztest`, confidence intervals |
| 10 | Effect size vs significance | Cohen's d visualised |
| 11 | Sample size planning | `TTestIndPower`, `NormalIndPower` |
| 12 | Practice | Paired t, ANOVA, A/B design |

### Golden rules

1. Always state H₀ and H₁ **before** looking at the data
2. Choose your α **before** running the test
3. Report **effect sizes alongside p-values** — a tiny p is not enough
4. Run a **power analysis before** collecting data to ensure your study is adequately sized
5. A non-significant result does not mean "no effect" — it means "insufficient evidence"